# Combined colocation, IDN and Veeva Link expert signal test (NBA 3.1)

One-feature-at-a-time signal checks for three combined ideas. No model.

1. **Expert in group**: another universe HCP in the HCP's practice (L1) or IDN (L4) is a Veeva expert.
2. **Expert peer writers**: among HCPs with a Kerendia-writing colleague, is the writing colleague a Veeva expert?
3. **Experts in low-adoption IDNs**: sizing of non-writing experts in IDNs where few colleagues write (brand list, not a model feature).

**Method** (same as `HCP_Colocation_data_assessment.md`): universe `HCP_Universe_without_call_hcps` (294,727); practice = BEST_AFFL `BRAND_RANK = 1` account; IDN = L4 of `mdm_stg_acct_hrchy_ker` ('NA' / 'Multiple Doctors' = no IDN); writers = LRx V7, `PERIOD_TYPE = 'W'`, Kerendia, NBRx > 0. Look-back 2025-07-25 to 2025-10-23, outcome 2025-10-24 to 2026-10-23. The HCP is always excluded from their own group. The new-writer view keeps only HCPs with no Kerendia NBRx in the look-back.

**Veeva expert**: best Kerendia-relevant decile (max of `CLINICAL_LEADER`, `FOCUS_AREA`, `DIGITAL_REACH_FOCUS_AREA` over the focus-name lists in `40_veeva_link_data_assessment_v2.md` section 5.3) of 8 to 10. Read directly from the UC federated tables `cph_db_prod.analytics_v2.anlt_veeva_link_*`.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np, re

C = "`ph-com-ai-us-prod-catalog-2553213839599380`"
UNIVERSE_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps"
LB_START, LB_END = "2025-07-25", "2025-10-23"     # look-back (feature) window, same as colocation assessment
OUT_START, OUT_END = "2025-10-24", "2026-10-23"   # outcome window
EXPERT_MIN_DECILE = 8                              # Veeva expert = best Kerendia-relevant decile 8-10
SEG_TABLE = f"{C}.kerendia_cp.cp_ker_hcp_2026q4"

univ = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
xw = (spark.table(f"{C}.kerendia_nba_3.source_provider_bhid_npi_mapping")
      .select("PROVIDER_ID", F.col("NPI_NUMBER").cast("string").alias("NPI"), "BH_ID"))
affl = (spark.table(f"{C}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static")
        .filter("BRAND_RANK = 1").select("BH_ID", F.col("BHO_ID").alias("L1_ID"), "HCP_SPECIALTY"))
hier = (spark.table(f"{C}.pace_ker_in_house_schema.mdm_stg_acct_hrchy_ker")
        .select(F.col("L1_CHILD_BHO_ID").alias("L1_ID"), "L4_IDN_BHO_ID", "L4_IDN_ORGNAME")
        .withColumn("L4_ID", F.when(F.col("L4_IDN_ORGNAME").isNull() |
                                    F.upper(F.trim("L4_IDN_ORGNAME")).isin("NA", "MULTIPLE DOCTORS"), None)
                               .otherwise(F.col("L4_IDN_BHO_ID")))
        .select("L1_ID", "L4_ID", "L4_IDN_ORGNAME"))

lrx = spark.table(f"{C}.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7") \
    .filter("PERIOD_TYPE = 'W' AND upper(PRODUCT_GROUP) = 'KERENDIA' AND NBRX > 0")
lb = lrx.filter(F.col("PERIOD_ID").between(LB_START, LB_END)).select("PROVIDER_ID").distinct().withColumn("LB_WRITER", F.lit(1))
out = lrx.filter(F.col("PERIOD_ID").between(OUT_START, OUT_END)).select("PROVIDER_ID").distinct().withColumn("OUT_WRITER", F.lit(1))

seg_cols = spark.table(SEG_TABLE).columns
print("segment table columns:", seg_cols)
seg = (spark.table(SEG_TABLE)
       .filter(F.col("FF").isin("CR1", "CR2", "CR4", "CR5") &
               ((F.upper("PRODUCT_1") == "KERENDIA") | (F.upper("PRODUCT_2") == "KERENDIA")))
       .select(F.col("BAYER_ID").cast("string").alias("BH_ID"), F.col("NOTE_2").alias("SEGMENT"))
       .dropDuplicates(["BH_ID"]))

base_sp = (univ.join(xw, "PROVIDER_ID", "left")
           .join(affl, "BH_ID", "left")
           .join(hier, "L1_ID", "left")
           .join(lb, "PROVIDER_ID", "left").join(out, "PROVIDER_ID", "left")
           .join(seg, "BH_ID", "left")
           .fillna(0, ["LB_WRITER", "OUT_WRITER"]))
base = base_sp.toPandas()
print("rows:", len(base), "| distinct PROVIDER_ID:", base.PROVIDER_ID.nunique())
print("hierarchy L1 rows vs distinct:", hier.count(), hier.select("L1_ID").distinct().count())
print("with L1:", base.L1_ID.notna().sum(), "| in real IDN:", base.L4_ID.notna().sum(), "| IDNs:", base.L4_ID.nunique())
print("outcome writers:", int(base.OUT_WRITER.sum()), f"({base.OUT_WRITER.mean():.1%})", "| look-back writers:", int(base.LB_WRITER.sum()))
print(base.SEGMENT.fillna("Non-target").value_counts())

In [0]:
# Focus-name lists read verbatim from assessment v2 section 5.3 (no retyping)
MD = "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Veeva Link To People Analysis/40_veeva_link_data_assessment_v2.md"
lines = open(MD, encoding="utf-8").read().splitlines()
i_cl = next(i for i, l in enumerate(lines) if l.startswith("#### Focus names used") and "CLINICAL_LEADER" in l)
i_fa = next(i for i, l in enumerate(lines) if l.startswith("#### Focus names used") and "FOCUS_AREA" in l)
i_end = next(i for i, l in enumerate(lines) if l.startswith("#### Coverage within the overlap"))
cl_names = [l.split("|")[3].strip() for l in lines[i_cl:i_fa]
            if l.startswith("| disease") or l.startswith("| drug_class")]
fa_names, dr_names = [], []
for l in lines[i_fa:i_end]:
    if l.startswith("|") and "(" in l.split("|")[1] and not l.startswith("| Theme") and not l.startswith("|---"):
        for n in l.split("|")[2].split("; "):
            n = n.strip()
            only_fa, only_dr = n.endswith("*"), n.endswith("\u2020")
            n = n.rstrip("*\u2020").strip()
            if not only_dr: fa_names.append(n)
            if not only_fa: dr_names.append(n)
fa_names, dr_names = sorted(set(fa_names)), sorted(set(dr_names))
print("focus names:", len(cl_names), len(fa_names), len(dr_names), "(assessment: 17 / 147 / 135)")

V = "cph_db_prod.analytics_v2.anlt_veeva_link_"
std = "SYS_TENANT = 'US' AND IS_LATEST_FLAG = 'Y' AND COALESCE(SYS_DELETE_FLAG, 0) = 0"
cl = (spark.table(V + "clinical_leader").filter(std).filter(F.col("TYPE").isin("disease", "drug_class"))
      .filter(F.col("FOCUS_NAME").isin(cl_names)).select("SE_ID", F.col("SE_DECILE").cast("int").alias("D")))
fa = (spark.table(V + "focus_area").filter(std).filter(F.col("FOCUS_NAME").isin(fa_names))
      .select("SE_ID", F.col("FOCUS_DECILE").cast("int").alias("D")))
dr = (spark.table(V + "digital_reach_focus_area").filter(std).filter(F.col("FOCUS_NAME").isin(dr_names))
      .select("SE_ID", F.col("SE_DECILE").cast("int").alias("D")))
best = cl.unionByName(fa).unionByName(dr).filter("D BETWEEN 1 AND 10").groupBy("SE_ID").agg(F.max("D").alias("BEST_DECILE"))
se = (spark.table(V + "scientific_expert")
      .filter("SYS_TENANT = 'US' AND COALESCE(ISDELETED, 0) = 0 AND NPI IS NOT NULL")
      .select("SE_ID", F.col("NPI").cast("string").alias("NPI")))
npis = spark.createDataFrame(base[["NPI"]].dropna().drop_duplicates())
veeva = (se.join(npis, "NPI").join(best, "SE_ID", "left")
         .groupBy("NPI").agg(F.max("BEST_DECILE").alias("BEST_DECILE"), F.lit(1).alias("IN_VEEVA"))).toPandas()

base = base.drop(columns=[c for c in ["BEST_DECILE", "IN_VEEVA"] if c in base.columns]).merge(veeva, on="NPI", how="left")
base["IN_VEEVA"] = base.IN_VEEVA.fillna(0).astype(int)
base["EXPERT"] = (base.BEST_DECILE >= EXPERT_MIN_DECILE).astype(int)
base["EXPERT9"] = (base.BEST_DECILE >= 9).astype(int)
print("matched to Veeva:", base.IN_VEEVA.sum(), f"({base.IN_VEEVA.mean():.1%})  | assessment 228,932 (77.7%)")
print("with in-scope decile:", base.BEST_DECILE.notna().sum(), " | assessment 147,489")
print("experts (decile 8-10):", base.EXPERT.sum(), f"({base.EXPERT.mean():.1%} of universe) | decile 9-10:", base.EXPERT9.sum())
print("expert rate own outcome:", round(100*base[base.EXPERT==1].OUT_WRITER.mean(),1), "vs non-expert", round(100*base[base.EXPERT==0].OUT_WRITER.mean(),1))

# Veeva load timing vs windows
for t in ["scientific_expert", "clinical_leader", "focus_area", "digital_reach_focus_area"]:
    r = spark.table(V + t).filter("SYS_TENANT = 'US' AND IS_LATEST_FLAG = 'Y'").agg(F.max("INSERT_TIMESTAMP").alias("max_insert"), F.max("SYS_CREATED_DT").alias("max_created")).collect()[0]
    print(t, "max INSERT_TIMESTAMP:", r.max_insert, "| max SYS_CREATED_DT:", r.max_created)

In [0]:
def add_peer(df, key, tag):
    g = df[key]
    has = g.notna()
    for col, src in [("N", None), ("W", "LB_WRITER"), ("E", "EXPERT"), ("EW", "EXW")]:
        s = pd.Series(1, index=df.index) if src is None else df[src]
        tot = s.groupby(g).transform("sum")
        df[f"PEER_{col}_{tag}"] = np.where(has, tot - s, np.nan)   # self excluded
    return df

base["EXW"] = base.EXPERT * base.LB_WRITER
base = add_peer(base, "L1_ID", "L1")
base = add_peer(base, "L4_ID", "L4")
# New-writer view = no Kerendia LRx activity at all (TRx > 0) in the look-back, as in the colocation assessment
lb_any = set(spark.table(f"{C}.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7")
             .filter("PERIOD_TYPE = 'W' AND upper(PRODUCT_GROUP) = 'KERENDIA' AND TRX > 0")
             .filter(F.col("PERIOD_ID").between(LB_START, LB_END))
             .select("PROVIDER_ID").distinct().toPandas().PROVIDER_ID)
base["NEW"] = (~base.PROVIDER_ID.isin(lb_any)).astype(int)
print("new-writer view HCPs:", base.NEW.sum(), "| baseline rate:", round(100 * base[base.NEW == 1].OUT_WRITER.mean(), 1), "(assessment about 4.4%)")

def rates(df, by):
    g = df.groupby(by, dropna=False, observed=True)
    out = g.agg(HCPS=("OUT_WRITER", "size"), RATE_ALL=("OUT_WRITER", "mean"))
    nw = df[df.NEW == 1].groupby(by, dropna=False, observed=True).agg(NEW_HCPS=("OUT_WRITER", "size"), RATE_NEW=("OUT_WRITER", "mean"))
    out = out.join(nw)
    out["RATE_ALL"] = (100 * out.RATE_ALL).round(1); out["RATE_NEW"] = (100 * out.RATE_NEW).round(1)
    return out.reset_index()

# Baseline reproduction (colocation assessment: L1 presence 15.7% vs 7.4%, new-writer 7.4% vs 4.4%)
l1p = base[base.PEER_N_L1 >= 1].copy()
l1p["L1_PEER_WRITER"] = (l1p.PEER_W_L1 > 0).astype(int)
print("HCPs with >=1 L1 peer:", len(l1p), "(assessment 233,102)")
print(rates(l1p, "L1_PEER_WRITER").to_string(index=False))
print("HCPs with >=1 L1 peer writer:", int((base.PEER_W_L1 > 0).sum()), "(assessment 65,122) | >=1 L4 peer writer:", int((base.PEER_W_L4 > 0).sum()), "(assessment 156,930)")

In [0]:
def size_band(n):
    return pd.cut(n, [0, 4, 19, np.inf], labels=["1-4 peers", "5-19 peers", "20+ peers"])

# 1a. L1 presence, among HCPs with >=1 L1 peer
l1p = base[base.PEER_N_L1 >= 1].copy()
l1p["L1_PEER_EXPERT"] = (l1p.PEER_E_L1 > 0).astype(int)
l1p["L1_PEER_WRITER"] = (l1p.PEER_W_L1 > 0).astype(int)
l1p["SIZE"] = size_band(l1p.PEER_N_L1)
print("1a. Expert colleague in practice (L1), HCPs with >=1 peer")
print(rates(l1p, "L1_PEER_EXPERT").to_string(index=False))
print("\n1a-own. Split by the HCP's own expert status")
print(rates(l1p, ["EXPERT", "L1_PEER_EXPERT"]).to_string(index=False))
print("\n1a-size. Within practice-size bands (confounder: bigger practices hold more experts)")
print(rates(l1p, ["SIZE", "L1_PEER_EXPERT"]).to_string(index=False))
print("median peers: expert colleague yes", l1p[l1p.L1_PEER_EXPERT == 1].PEER_N_L1.median(), "| no", l1p[l1p.L1_PEER_EXPERT == 0].PEER_N_L1.median())

# 1b. Does it add beyond the practice peer-writer flag? (2x2)
print("\n1b. Practice peer writer x practice expert colleague")
print(rates(l1p, ["L1_PEER_WRITER", "L1_PEER_EXPERT"]).to_string(index=False))
print("\n1b-size. Same 2x2 within size bands")
print(rates(l1p, ["SIZE", "L1_PEER_WRITER", "L1_PEER_EXPERT"]).to_string(index=False))

# 1c. L4 IDN: presence and expert share (zero band + quintiles of non-zero share)
l4p = base[base.PEER_N_L4 >= 1].copy()
l4p["L4_PEER_EXPERT"] = (l4p.PEER_E_L4 > 0).astype(int)
l4p["L4_EXPERT_SHARE"] = l4p.PEER_E_L4 / l4p.PEER_N_L4
l4p["L4_WRITER_SHARE"] = l4p.PEER_W_L4 / l4p.PEER_N_L4
print("\n1c. Expert colleague in IDN (L4), HCPs with >=1 IDN peer:", len(l4p))
print(rates(l4p, "L4_PEER_EXPERT").to_string(index=False))
nz = l4p.L4_EXPERT_SHARE > 0
l4p["EXPERT_SHARE_Q"] = "0 (no expert peer)"
l4p.loc[nz, "EXPERT_SHARE_Q"] = pd.qcut(l4p.loc[nz, "L4_EXPERT_SHARE"].rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"]).astype(str)
print("\n1c-share. IDN expert share: zero band + quintiles of non-zero share")
r = rates(l4p, "EXPERT_SHARE_Q")
r["SHARE_RANGE"] = r.EXPERT_SHARE_Q.map(l4p.groupby("EXPERT_SHARE_Q").L4_EXPERT_SHARE.agg(lambda s: f"{s.min():.1%}-{s.max():.1%}"))
print(r.to_string(index=False))
print("\n1c-own. IDN expert share band by own expert status")
print(rates(l4p, ["EXPERT", "EXPERT_SHARE_Q"]).to_string(index=False))

# 1d. Does IDN expert share add beyond IDN writer share? Top-fifth x top-fifth
l4p["TOP_WRITER_SHARE"] = (l4p.L4_WRITER_SHARE >= l4p.L4_WRITER_SHARE.quantile(0.8)).astype(int)
l4p["TOP_EXPERT_SHARE"] = (l4p.L4_EXPERT_SHARE >= l4p.L4_EXPERT_SHARE.quantile(0.8)).astype(int)
print("\n1d. IDN top-fifth writer share x top-fifth expert share")
print(rates(l4p, ["TOP_WRITER_SHARE", "TOP_EXPERT_SHARE"]).to_string(index=False))
print("spearman(writer share, expert share) =", round(l4p[["L4_WRITER_SHARE", "L4_EXPERT_SHARE"]].corr(method="spearman").iloc[0, 1], 3))

In [0]:
# 2a. L1: among HCPs with >=1 practice peer writer
w1 = base[base.PEER_W_L1 >= 1].copy()
w1["EXPERT_PEER_WRITER"] = (w1.PEER_EW_L1 > 0).astype(int)
w1["N_PEER_WRITERS"] = pd.cut(w1.PEER_W_L1, [0, 1, 3, np.inf], labels=["1 writer", "2-3 writers", "4+ writers"])
print("2a. Practice (L1), HCPs with >=1 peer writer:", len(w1))
print(rates(w1, "EXPERT_PEER_WRITER").to_string(index=False))
print("\n2a-count. Controlling for number of peer writers (expert writer goes with more writers)")
print(rates(w1, ["N_PEER_WRITERS", "EXPERT_PEER_WRITER"]).to_string(index=False))
print("\n2a-own. Split by own expert status")
print(rates(w1, ["EXPERT", "EXPERT_PEER_WRITER"]).to_string(index=False))

# 2b. L4: expert share of peer writers, among HCPs with >=1 IDN peer writer
w4 = base[base.PEER_W_L4 >= 1].copy()
w4["EXPERT_SHARE_OF_WRITERS"] = w4.PEER_EW_L4 / w4.PEER_W_L4
w4["EXW_BAND"] = pd.cut(w4.EXPERT_SHARE_OF_WRITERS, [-0.001, 0, 0.2, 0.4, 1.0], labels=["0%", "0-20%", "20-40%", "40%+"])
w4["WRITER_SHARE_Q"] = pd.qcut((w4.PEER_W_L4 / w4.PEER_N_L4).rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
print("\n2b. IDN (L4), HCPs with >=1 IDN peer writer:", len(w4))
print(rates(w4, "EXW_BAND").to_string(index=False))
print("\n2b-ctrl. Within IDN writer-share quintiles")
print(rates(w4, ["WRITER_SHARE_Q", "EXW_BAND"]).to_string(index=False))

In [0]:
ex = base[(base.EXPERT == 1) & (base.LB_WRITER == 0) & (base.PEER_N_L4 >= 1)].copy()
ex["IDN_WRITER_SHARE"] = ex.PEER_W_L4 / ex.PEER_N_L4
ex["ADOPTION_BAND"] = pd.cut(ex.IDN_WRITER_SHARE, [-0.001, 0, 0.05, 0.10, 0.20, 1.0],
                             labels=["0%", "0-5%", "5-10%", "10-20%", ">20%"])
print("3a. Universe experts (decile 8-10) with no Kerendia NBRx in look-back, in a real IDN with >=1 peer:", len(ex))
print("    all universe experts:", int(base.EXPERT.sum()), "| non-writing:", int(((base.EXPERT == 1) & (base.LB_WRITER == 0)).sum()))
print(rates(ex, "ADOPTION_BAND").to_string(index=False))

low = ex[ex.IDN_WRITER_SHARE <= 0.05]
print("\n3b. Low-adoption (IDN writer share <= 5%):", len(low), "experts in", low.L4_ID.nunique(), "IDNs")
print(low.SEGMENT.fillna("Non-target").value_counts().to_string())
print("\n3c. Top specialties (BEST_AFFL HCP_SPECIALTY)")
print(low.HCP_SPECIALTY.fillna("Unknown").value_counts().head(10).to_string())
print("\n3d. Largest low-adoption IDNs by number of non-writing experts")
print(low.groupby("L4_IDN_ORGNAME").agg(EXPERTS=("PROVIDER_ID", "size"), IDN_PEERS=("PEER_N_L4", "max"),
      WRITER_SHARE=("IDN_WRITER_SHARE", "mean")).sort_values("EXPERTS", ascending=False).head(10).round(3).to_string())

In [0]:
# V1. Recount L1 expert-colleague presence with a Spark self-join (independent of the pandas transform)
bsp = spark.createDataFrame(base[["PROVIDER_ID", "L1_ID", "EXPERT", "LB_WRITER", "OUT_WRITER"]].dropna(subset=["L1_ID"]))
bsp.createOrReplaceTempView("v_base")
v1 = spark.sql("""
  SELECT a.PROVIDER_ID, MAX(CASE WHEN b.EXPERT = 1 THEN 1 ELSE 0 END) AS HAS_EXPERT_PEER, MAX(a.OUT_WRITER) AS OUT_WRITER
  FROM v_base a JOIN v_base b ON a.L1_ID = b.L1_ID AND a.PROVIDER_ID <> b.PROVIDER_ID
  GROUP BY a.PROVIDER_ID""").groupBy("HAS_EXPERT_PEER").agg(F.count("*").alias("HCPS"), F.round(100 * F.avg("OUT_WRITER"), 1).alias("RATE_ALL")).orderBy("HAS_EXPERT_PEER")
print("V1 self-join recount (compare with 1a):"); print(v1.toPandas().to_string(index=False))

# V2. Partitions add up
print("V2 1b cells sum to L1 base:", rates(l1p, ["L1_PEER_WRITER", "L1_PEER_EXPERT"]).HCPS.sum() == len(l1p), "| 3a bands sum:", rates(ex, "ADOPTION_BAND").HCPS.sum() == len(ex))

# V3. Self-exclusion on a toy practice: 3 HCPs, only HCP 'a' is an expert writer
toy = pd.DataFrame({"L1_ID": ["P", "P", "P"], "LB_WRITER": [1, 0, 0], "EXPERT": [1, 0, 0]}); toy["EXW"] = toy.EXPERT * toy.LB_WRITER
toy = add_peer(toy, "L1_ID", "L1")
print("V3 toy peers (expect a: E=0,W=0; b,c: E=1,W=1,EW=1):"); print(toy[["PEER_N_L1", "PEER_W_L1", "PEER_E_L1", "PEER_EW_L1"]].to_string())

# V4. Windows do not overlap and Veeva loads precede the look-back
print("V4 look-back ends", LB_END, "< outcome starts", OUT_START, "->", LB_END < OUT_START)